In [ ]:
from pathlib import Path
from glob import glob
import datetime as dt
import json
import sys

# Launch Jupyter from this distribution folder.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "default_config.json").is_file() and (p / "wheels").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the div_free_rbf distribution folder")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from IPython.display import display

try:
    from rbf_fit import DFFConfig, fit_files
except ModuleNotFoundError as exc:
    if exc.name != "rbf_fit":
        raise
    raise ModuleNotFoundError(
        "Install wheels/*.whl in this notebook kernel's Python environment; "
        f"current Python: {sys.executable}"
    ) from exc
print("Kernel Python:", sys.executable)


In [ ]:
# Edit these patterns to select an event, time block, or radar.
# Relative patterns are resolved from ROOT; absolute paths also work.
# Use ** for recursive matching, e.g. "demo_data/fitacf/**/*.fitacf*".
FITACF_GLOB = "demo_data/fitacf/20220410.*fitacf3*"
MAP2_GLOB = "demo_data/map2/20220410.*map2*"


def matching_files(pattern):
    expanded = Path(pattern).expanduser()
    resolved_pattern = expanded if expanded.is_absolute() else ROOT / expanded
    return sorted({Path(p).resolve() for p in glob(str(resolved_pattern), recursive=True)
                   if Path(p).is_file()})


FITACF_FILES = matching_files(FITACF_GLOB)
map_files = matching_files(MAP2_GLOB)
if not FITACF_FILES:
    raise FileNotFoundError(f"No fitacf files matched {FITACF_GLOB!r} (root: {ROOT})")
if len(map_files) != 1:
    raise ValueError(
        f"MAP2_GLOB must match exactly one file; matched {len(map_files)}: {map_files}. "
        "Narrow the pattern to the event date and hemisphere."
    )
MAP2_FILE = map_files[0]
FIT_TIME = "2022-04-10T05:00:00Z"  # change to match your files
HEMISPHERE = "north"             # "north" or "south"
CONFIG_FILE = ROOT / "default_config.json"
OUTPUT_DIR = ROOT / "outputs" / "demo"
MIN_ABS_LATITUDE = 50.0
REFERENCE_SPEED = 500.0          # m/s; same arrow scale in both panels
ARROW_LENGTH_DEGREES = 3.0       # displayed length for REFERENCE_SPEED

config = DFFConfig.load(CONFIG_FILE)
print("Configuration:", config.source)
print("fitacf inputs:", *FITACF_FILES, sep="\n  ")
print("MAP2 input:", MAP2_FILE)
print(json.dumps(config.values, indent=2))


In [ ]:
result = fit_files(FITACF_FILES, MAP2_FILE, FIT_TIME,
                   config=config, hemisphere=HEMISPHERE)
# Preserve the actual observations used in conditioning, including signed LOS.
los = result.fitter.data_df.copy(deep=True)
dff = result.data.copy(deep=True)
print(f"Selected LOS measurements: {len(los):,}; DFF vectors: {len(dff):,}")
display(los[["date", "radar", "mlats", "mlt", "vel", "mag_azm_los"]].head())
display(dff.head())


In [ ]:
def plot_components(ax, latitude, mlt, east, north, title, norm, *, colors=None, cmap="viridis"):
    latitude, mlt, east, north = np.broadcast_arrays(
        *[np.asarray(v, dtype=float) for v in (latitude, mlt, east, north)])
    colors = np.hypot(east, north) if colors is None else np.asarray(colors, dtype=float)
    valid = np.isfinite(latitude + mlt + east + north) & np.isfinite(colors)
    valid &= (np.abs(latitude) >= MIN_ABS_LATITUDE) & (np.abs(latitude) <= 90)
    valid &= latitude >= 0 if HEMISPHERE == "north" else latitude <= 0
    mlt = np.mod(mlt, 24)
    valid &= (mlt >= 18) | np.isclose(mlt, 0, rtol=0, atol=1e-8)
    latitude, mlt, east, north = [v[valid] for v in (latitude, mlt, east, north)]
    if not latitude.size:
        raise ValueError(f"No vectors within plotting bounds for {title}")
    colors = colors[valid]
    theta = 2 * np.pi * mlt / 24
    radius = 90 - np.abs(latitude)
    poleward = np.sign(latitude) * north
    x, y = radius * np.sin(theta), -radius * np.cos(theta)
    u = east * np.cos(theta) - poleward * np.sin(theta)
    v = east * np.sin(theta) + poleward * np.cos(theta)
    quiver = ax.quiver(x, y, u, v, colors, cmap=cmap,
                       norm=norm, angles="xy", scale_units="xy",
                       scale=REFERENCE_SPEED / ARROW_LENGTH_DEGREES, width=0.003)
    angle = np.linspace(1.5 * np.pi, 2 * np.pi, 181)
    limit = 90 - MIN_ABS_LATITUDE
    for lat in np.arange(MIN_ABS_LATITUDE, 90, 10):
        r = 90 - lat
        ax.plot(r * np.sin(angle), -r * np.cos(angle), color="0.8", lw=0.7)
        ax.text(0.5, -r, f"{lat:.0f}°", fontsize=8, color="0.4")
    for hour in (18, 21, 24):
        a = 2 * np.pi * hour / 24
        ax.plot([0, limit * np.sin(a)], [0, -limit * np.cos(a)], color="0.9", lw=0.6)
        ax.text((limit + 3) * np.sin(a), -(limit + 3) * np.cos(a),
                f"{hour % 24:02d}", ha="center", va="center")
    ax.quiverkey(quiver, 0.78, -0.03, REFERENCE_SPEED, f"{REFERENCE_SPEED:g} m/s",
                 coordinates="axes", labelpos="E")
    ax.set(xlim=(-limit-6, 6), ylim=(-limit-6, 6),
           aspect="equal", title=f"{title}")
    ax.axis("off")
    return quiver

In [ ]:
los_limit = float(config.section("plot")["los_color_limit"])
if not np.isfinite(los_limit) or los_limit <= 0:
    raise ValueError("plot.los_color_limit must be positive and finite")
los_norm = Normalize(vmin=-los_limit, vmax=los_limit)
utc_time = dt.datetime.fromisoformat(FIT_TIME)
if utc_time.tzinfo is not None:
    utc_time = utc_time.astimezone(dt.timezone.utc)

if not hasattr(result.fitter, "pre_filter_data"):
    raise RuntimeError("Restart the kernel and rerun the fitting cell to capture pre-filter LOS data")
pre_los = result.fitter.pre_filter_data.copy(deep=True)
post_los = result.fitter.data_df.copy(deep=True)
fig_filters, filter_axes = plt.subplots(1, 2, figsize=(13, 6.5), layout="constrained")

for ax, measurements, title in zip(
    filter_axes, (pre_los, post_los), ("LOS before quality filters", "LOS after quality filters")
):
    azimuth = measurements["mag_azm_los"].to_numpy()
    signed_velocity = measurements["vel"].to_numpy()
    q_filter = plot_components(
        ax, measurements["mlats"], measurements["mlt"],
        signed_velocity * np.sin(azimuth), signed_velocity * np.cos(azimuth),
        title, los_norm, colors=signed_velocity, cmap="RdBu",
    )
fig_filters.colorbar(q_filter, ax=filter_axes, shrink=0.75,
                    label="Signed LOS velocity (m/s)", extend="both")
fig_filters.suptitle(f"{utc_time:%Y-%m-%d %H:%M:%S} UTC | LOS filtering comparison")
plt.show()


In [ ]:
los_limit = float(config.section("plot")["los_color_limit"])
if not np.isfinite(los_limit) or los_limit <= 0:
    raise ValueError("plot.los_color_limit must be positive and finite")
los_norm = Normalize(vmin=-los_limit, vmax=los_limit)
utc_time = dt.datetime.fromisoformat(FIT_TIME)
if utc_time.tzinfo is not None:
    utc_time = utc_time.astimezone(dt.timezone.utc)

plot_dff = dff.copy()
if "fit_source" in plot_dff:
    plot_dff = plot_dff.loc[plot_dff["fit_source"].ne("map_prior")]
speeds = np.r_[los["vel"].abs(), np.hypot(plot_dff["east_vel"], plot_dff["north_vel"])]
finite_speeds = speeds[np.isfinite(speeds)]
if not finite_speeds.size:
    raise ValueError("No finite velocities to plot")
norm = Normalize(vmin=0, vmax=max(1, float(np.quantile(finite_speeds, 0.98))))

if not np.isfinite(los_limit) or los_limit <= 0:
    raise ValueError("plot.los_color_limit must be positive and finite")

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), layout="constrained")
los_azimuth = los["mag_azm_los"].to_numpy()

q_los = plot_components(axes[0], los["mlats"], los["mlt"],
                los["vel"] * np.sin(los_azimuth),
                los["vel"] * np.cos(los_azimuth), "Measured LOS components", los_norm,
                colors=los["vel"], cmap="RdBu")

q = plot_components(axes[1], plot_dff["mlat"], plot_dff["mlt"],
                    plot_dff["east_vel"], plot_dff["north_vel"], "DFF vectors", norm)
fig.colorbar(q_los, ax=axes[0], shrink=0.75, label="LOS velocity (m/s)", extend="both")
fig.colorbar(q, ax=axes[1], shrink=0.75, label="DFF speed (m/s)", extend="max")
utc_time = dt.datetime.fromisoformat(FIT_TIME)
if utc_time.tzinfo is not None:
    utc_time = utc_time.astimezone(dt.timezone.utc)
fig.suptitle(f"{utc_time:%Y-%m-%d %H:%M:%S} UTC")
plt.show()
fig.savefig("dff_fit_demo.png", dpi=200)

## Save outputs and run settings

The LOS CSV records the cleaned input measurements. 
The DFF CSV contains fitted vectors. 

In [ ]:

# los.to_csv("selected_los.csv")
# dff.to_csv("dff_vectors.csv")

